In [1]:
import numpy as np
from scipy.special import erf, j0, gammaln, expi

In [2]:
def integMonteCarlo(func, dim, limInf, limSup, numMuestras, semilla=None):
    genAleat = np.random.default_rng(semilla)
    vol = np.prod(limSup - limInf)
    sumaF = 0.0
    sumaF2 = 0.0

    for _ in range(numMuestras):
        punto = limInf + genAleat.random(dim) * (limSup - limInf)
        valor = func(punto)
        sumaF += valor
        sumaF2 += valor * valor

    promF = sumaF / numMuestras
    promF2 = sumaF2 / numMuestras
    varianza = (promF2 - promF**2) / numMuestras
    errEst = vol * np.sqrt(max(varianza, 0.0))
    intEst = vol * promF

    return intEst, errEst

In [3]:
# Caso 1: Integral de erf(x) en [0, 1] (1D)
# erf no tiene antiderivada elemental
def func1(x):
    return erf(x[0])

res1, err1 = integMonteCarlo(func1, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([1.0]), numMuestras=100000, semilla=42)
print(f"Caso 1 - erf(x) en [0,1]: {res1:.6f} ± {err1:.6f}")


# Caso 2: Integral de J₀(x) en [0, 5] (1D)
# Función de Bessel de primera especie, sin primitiva elemental
def func2(x):
    return j0(x[0])

res2, err2 = integMonteCarlo(func2, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([5.0]), numMuestras=100000, semilla=42)
print(f"Caso 2 - J₀(x) en [0,5]: {res2:.6f} ± {err2:.6f}")


# Caso 3: Integral de ln(Γ(x)) en [1, 3] (1D)
# Log-gamma, función especial sin antiderivada elemental
def func3(x):
    return gammaln(x[0])

res3, err3 = integMonteCarlo(func3, dim=1, limInf=np.array([1.0]), 
                              limSup=np.array([3.0]), numMuestras=100000, semilla=42)
print(f"Caso 3 - ln(Γ(x)) en [1,3]: {res3:.6f} ± {err3:.6f}")


# Caso 4: Integral doble de exp(-x²-y²)·cos(xy) en [0,1]×[0,1] (2D)
# Producto de gaussiana y oscilación, sin primitiva cerrada
def func4(x):
    return np.exp(-(x[0]**2 + x[1]**2)) * np.cos(x[0] * x[1])

res4, err4 = integMonteCarlo(func4, dim=2, limInf=np.array([0.0, 0.0]), 
                              limSup=np.array([1.0, 1.0]), numMuestras=200000, semilla=42)
print(f"Caso 4 - exp(-x²-y²)·cos(xy) en [0,1]²: {res4:.6f} ± {err4:.6f}")


# Caso 5: Integral triple de sin(x·y·z) / (1 + x² + y² + z²) en [0,2]³ (3D)
# Función oscilatoria racional, sin antiderivada elemental en 3D
def func5(x):
    xyz = x[0] * x[1] * x[2]
    denom = 1.0 + x[0]**2 + x[1]**2 + x[2]**2
    return np.sin(xyz) / denom if denom > 1e-10 else 0.0

res5, err5 = integMonteCarlo(func5, dim=3, limInf=np.array([0.0, 0.0, 0.0]), 
                              limSup=np.array([2.0, 2.0, 2.0]), numMuestras=300000, semilla=42)
print(f"Caso 5 - sin(xyz)/(1+x²+y²+z²) en [0,2]³: {res5:.6f} ± {err5:.6f}")

Caso 1 - erf(x) en [0,1]: 0.486647 ± 0.000788
Caso 2 - J₀(x) en [0,5]: 0.707504 ± 0.008013
Caso 3 - ln(Γ(x)) en [1,3]: 0.224950 ± 0.001527
Caso 4 - exp(-x²-y²)·cos(xy) en [0,1]²: 0.540373 ± 0.000516
Caso 5 - sin(xyz)/(1+x²+y²+z²) en [0,2]³: 0.646702 ± 0.001013
